In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt
import librosa
import soundfile as sf
import os
import json
import json5
import scipy
from scipy import signal
from math import pi
import math
from typing import Optional
from importlib.resources import files
from omegaconf import OmegaConf
import torch.nn.functional as F
from torch import nn
import random
random.seed(114)
import pandas as pd
from glob import glob

In [2]:
class StreamingMean:
    def __init__(self):
        self.mean = None
        self.n = 0
        
    def update(self, x: np.ndarray):
        """传入新的 np.array 来更新均值"""
        self.n += 1
        if self.mean is None:
            # 第一笔数据直接作为初始均值
            self.mean = x.copy()
        else:
            # 动态更新公式
            self.mean += (x - self.mean) / self.n

In [3]:
ckpt_path = '/data/250010171/code/EigeNet_discriminant/ckpts/discriminant/base2_g2_align2_debug_layer6/checkpoint/epoch-0009_step-0007400_loss-2.084283/pytorch_model.bin'
state_dict = torch.load(ckpt_path, map_location="cpu")

/tmp/ipykernel_62486/3527124940.py:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  state_dict = torch.load(ckpt_path, map_location="cpu")


In [ ]:
main_parms = 0
align_parms = 0
for key, value in state_dict.items():
    if key.startswith('align'):
        align_parms += value.numel()
    else:
        main_parms += value.numel()
main_parms = round(main_parms / 1e6, 2)
align_parms = round(align_parms / 1e6, 2)
print(f"main_parms: {main_parms}M, align_parms: {align_parms}M")



main_parms: 116.27M, align_parms: 5.25M


In [7]:
root = '/mnt/data/jingchong/eigenet/output/'
all_models = os.listdir(root)
all_model_paths = [os.path.join(root, model) for model in all_models]
for model_path in all_model_paths:
    ckpt = os.listdir(model_path)
    ckpt_path = os.path.join(model_path, ckpt[0])
    audio_dir = os.path.join(ckpt_path, 'unseen')
    print(f"model: {model_path}")
    for K in os.listdir(audio_dir):
        audio_num = len(glob(os.path.join(audio_dir, K, '*.wav')))
        print(f"K: {K}, audio_num: {audio_num}")


model: /mnt/data/jingchong/eigenet/output/discriminant_ablation_ca_noalign
K: 1, audio_num: 4836
K: 4, audio_num: 4836
K: 8, audio_num: 4316
model: /mnt/data/jingchong/eigenet/output/discriminant_ablation_only_depth_noalign
K: 1, audio_num: 4836
K: 4, audio_num: 4836
K: 8, audio_num: 4316
model: /mnt/data/jingchong/eigenet/output/discriminant_ablation_only_loc_noalign
K: 1, audio_num: 4836
K: 4, audio_num: 4836
K: 8, audio_num: 4316
model: /mnt/data/jingchong/eigenet/output/discriminant_ablation_sa_noalign
K: 1, audio_num: 4836
K: 4, audio_num: 4836
K: 8, audio_num: 4316
model: /mnt/data/jingchong/eigenet/output/discriminant_base2_g2_align2_debug_layer6
K: 1, audio_num: 4836
K: 2, audio_num: 1160
K: 4, audio_num: 4800
K: 8, audio_num: 4316
model: /mnt/data/jingchong/eigenet/output/discriminant_base2_g2_noalign_debug
K: 8, audio_num: 4316


In [4]:
rir_path = '/data/share/amphion/data/noise-and-rirs/haa/classroomBase/RIRs.npy'
rir_npy = np.load(rir_path)
print(rir_npy.shape)
xyzs_path = '/data/share/amphion/data/noise-and-rirs/haa/classroomBase/xyzs.npy'
xyzs_npy = np.load(xyzs_path)
print(xyzs_npy.shape)


(630, 671884)
(630, 3)


In [5]:
print(rir_npy[0])
print(xyzs_npy[0])

[ 6.11514818e-06  7.35741694e-06  5.00817302e-06 ... -7.26507135e-07
  3.53323294e-07  5.90716509e-07]
[2.4241125 0.581025  0.32     ]
